# SFT v1 — Qwen3.5-4B LoRA ince ayar

TUS çoktan seçmeli verisiyle ilk ince ayar turu.

**Çalıştırma:** Colab, Runtime → GPU (A100 ya da L4). Hücreleri sırayla çalıştır. Eğitim ~1,5 saat.

**Ön koşul:** eğitim verisi `MyDrive/tus_data/` altında zip olarak. Veri depoda yok — `dataset/README.md`'ye bak.

**Sonuç:** doğruluk 62,3% → 70,9% (p = 0,003). Talimat takibi bozuldu; v2 bunu düzeltiyor.

## 01 kurulum

`hucreler/v1/01_kurulum.py`

In [ ]:
%%capture
!pip install -q unsloth

## 02 drive bagla

`hucreler/v1/02_drive_bagla.py`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 03 zip ac

`hucreler/v1/03_zip_ac.py`

In [ ]:
import zipfile, glob, os
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/tus_data/TUS_Dataset_Final-20261002T154401Z-1-001.zip"
HEDEF    = "/content/tus_veri"

if not glob.glob(f"{HEDEF}/**/egitim.jsonl", recursive=True):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(HEDEF)

bulunan = glob.glob(f"{HEDEF}/**/egitim.jsonl", recursive=True)
assert bulunan, f"egitim.jsonl bulunamadi. Cikan icerik: {os.listdir(HEDEF)[:10]}"
DATA_DIR = str(Path(bulunan[0]).parent)
print("DATA_DIR:", DATA_DIR)
for f in ("egitim.jsonl", "test_cevap.jsonl", "test_aciklamali.jsonl"):
    p = Path(DATA_DIR) / f
    assert p.exists(), f"eksik dosya: {f}"
    print(f"  {f}: {sum(1 for _ in open(p, encoding='utf-8'))} kayit")

## 04 ayarlar

`hucreler/v1/04_ayarlar.py`

In [ ]:
SMOKE = True

MODEL_NAME = "Qwen/Qwen3.5-4B"
OUT_DIR    = "/content/drive/MyDrive/tus_sft/qwen35_4b" + ("_smoke" if SMOKE else "")

MAX_SEQ_LEN    = 2048
MAX_EXPL_CHARS = 3000
EPOCHS         = 1 if SMOKE else 2
SEED           = 42

SYSTEM_PROMPT = (
    "Sen bir tıp uzmanısın. Aşağıdaki TUS sorusunu çöz. "
    "İlk satıra 'Doğru cevap: <harf>) <şık metni>' yaz, ardından kısa bir açıklama ver."
)

import torch, os
gpu = torch.cuda.get_device_name(0)
if   "A100" in gpu or "H100" in gpu: BS, GA, LOAD_4BIT = 8, 2, False
elif "L4"   in gpu or "A10"  in gpu: BS, GA, LOAD_4BIT = 4, 4, False
else:                                BS, GA, LOAD_4BIT = 2, 8, True
print(f"{gpu} | batch {BS} x grad_accum {GA} = {BS*GA} | 4bit={LOAD_4BIT}")

## 05 veri

`hucreler/v1/05_veri.py`

In [ ]:
import json, re, random, unicodedata
from collections import Counter

def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

THINK  = re.compile(r"<think>\n?(.*?)\n?</think>", re.S)
ANSWER = re.compile(r"Doğru cevap:\s*([A-E])")
OPTION = re.compile(r"^([A-E])\) (.+)$", re.M)

def option_text(question, letter):
    for h, t in OPTION.findall(question):
        if h == letter:
            return t.strip()
    return ""

def shorten(text, limit=MAX_EXPL_CHARS):
    if len(text) <= limit:
        return text
    cut = text[:limit]
    for sep in ("\n\n", "\n", ". "):
        i = cut.rfind(sep)
        if i > limit * 0.5:
            return cut[:i + len(sep)].strip()
    return cut.strip()

def usable(rec):
    q, a = rec["messages"][0]["content"], rec["messages"][-1]["content"]
    m = ANSWER.search(a)
    if not m:
        return False
    letters = [h for h, _ in OPTION.findall(q)]
    if letters not in (list("ABCDE"), list("ABCD")) or m.group(1) not in letters:
        return False
    th = THINK.search(a)
    return bool(th and len(th.group(1).strip()) >= 40)

def to_answer_first(rec):
    q = rec["messages"][0]["content"]
    a = rec["messages"][-1]["content"]
    letter = ANSWER.search(a).group(1)
    expl   = shorten(THINK.search(a).group(1).strip())
    opt    = option_text(q, letter)
    head   = f"Doğru cevap: {letter}) {opt}" if opt else f"Doğru cevap: {letter}"
    return {"question": q,
            "answer": f"{head}\n\nAçıklama: {expl}",
            "letter": letter,
            "group": (rec.get("meta") or {}).get("group", "?")}

raw  = load_jsonl(f"{DATA_DIR}/egitim.jsonl")
kept = [r for r in raw if usable(r)]
rows = [to_answer_first(r) for r in kept]
print(f"egitim.jsonl {len(raw)} -> kullanilabilir {len(rows)} (elenen {len(raw)-len(rows)})")

random.Random(SEED).shuffle(rows)
n_val = max(1, int(len(rows) * 0.02))
val_rows, train_rows = rows[:n_val], rows[n_val:]

test_acc = [{"question": r["soru"], "letter": r["dogru_cevap"],
             "group": (r.get("meta") or {}).get("group", "?")}
            for r in load_jsonl(f"{DATA_DIR}/test_cevap.jsonl")]

test_expl = [{"question": r["messages"][0]["content"],
              "letter": ANSWER.search(r["messages"][-1]["content"]).group(1),
              "reference": THINK.search(r["messages"][-1]["content"]).group(1).strip(),
              "group": (r.get("meta") or {}).get("group", "?")}
             for r in load_jsonl(f"{DATA_DIR}/test_aciklamali.jsonl")]

if SMOKE:
    test_acc  = random.Random(SEED).sample(test_acc, 40)
    test_expl = random.Random(SEED).sample(test_expl, 10)

def key(s):
    s = unicodedata.normalize("NFKD", s.lower()).translate(str.maketrans("çğıöşüâîû", "cgiosuaiu"))
    return frozenset(w for w in re.findall(r"[a-z]{4,}", s.split("\n")[0]))

train_keys = [key(r["question"]) for r in rows]
index = {}
for i, k in enumerate(train_keys):
    for w in k:
        index.setdefault(w, []).append(i)

def leaks(q):
    t = key(q)
    if len(t) < 4:
        return False
    hits = Counter()
    for w in t:
        for i in index.get(w, ()):
            hits[i] += 1
    return any(c / len(t | train_keys[i]) >= 0.8 for i, c in hits.most_common(3))

sizan = sum(leaks(r["question"]) for r in test_acc) + sum(leaks(r["question"]) for r in test_expl)
assert sizan == 0, f"SIZINTI: {sizan} test sorusu egitim setinde de var — koşuyu durdur"
print(f"train {len(train_rows)} | val {len(val_rows)} | test(dogruluk) {len(test_acc)} | test(aciklama) {len(test_expl)}")
print("sizinti kontrolu: temiz")

## 06 model

`hucreler/v1/06_model.py`

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=LOAD_4BIT,
)

tok = getattr(tokenizer, "tokenizer", tokenizer)
END = "<|im_end|>"
assert tok.convert_tokens_to_ids(END) != tok.unk_token_id, "Qwen <|im_end|> tokeni bulunamadi"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def build_prompt(question):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question}]
    return tok.apply_chat_template(msgs, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)

ornek = build_prompt("Soru?")
print(ornek)
assert "<think>" in ornek, "no-think sablonunda bos think blogu bekleniyordu"

## 07 degerlendirme

`hucreler/v1/07_degerlendirme.py`

In [ ]:
BASLIK = re.compile(r"^\s*Doğru cevap\s*:?\s*\(?([A-E])\)?")
HERHANGI = re.compile(r"(?:Doğru\s*)?(?:cevap|yanıt|answer)\s*[:\-]?\s*\(?([A-E])\)?", re.I)

def strip_think(t):
    return re.sub(r"<think>.*?</think>", "", t, flags=re.S).strip()

def parse_strict(t):
    m = BASLIK.match(strip_think(t))
    return m.group(1).upper() if m else None

def parse_loose(t):
    c = strip_think(t)
    m = BASLIK.match(c) or HERHANGI.search(c)
    if m:
        return m.group(1).upper()
    m = re.search(r"\b([A-E])\)", c)
    return m.group(1).upper() if m else None

@torch.no_grad()
def evaluate(rows, max_new_tokens=64, batch_size=16, etiket=""):
    FastLanguageModel.for_inference(model)
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        enc = tok([build_prompt(r["question"]) for r in batch],
                  return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

    s_pred = [parse_strict(o) for o in outs]
    l_pred = [parse_loose(o)  for o in outs]
    gold   = [r["letter"] for r in rows]
    s_acc  = sum(p == g for p, g in zip(s_pred, gold)) / len(rows)
    l_acc  = sum(p == g for p, g in zip(l_pred, gold)) / len(rows)
    uyum   = sum(p is not None for p in s_pred) / len(rows)
    yok    = sum(p is None for p in l_pred)

    print(f"{etiket}dogruluk  kati {s_acc:.1%} | gevsek {l_acc:.1%}")
    print(f"{etiket}bicime uyum {uyum:.1%} | hic cevap vermeyen {yok}/{len(rows)}")

    per = {}
    for r, p in zip(rows, l_pred):
        g = r.get("group", "?")
        d, t = per.get(g, (0, 0))
        per[g] = (d + (p == r["letter"]), t + 1)
    if len(per) > 1:
        print(f"{etiket}kaynak bazinda:")
        for g, (d, t) in sorted(per.items(), key=lambda x: -x[1][1])[:6]:
            print(f"{etiket}  {g:<34} {d}/{t}  {d/t:.0%}")
    return {"strict": s_acc, "loose": l_acc, "format": uyum, "outs": outs, "per_group": per}

## 08 taban cizgisi egitimsiz mode

`hucreler/v1/08_taban_cizgisi_egitimsiz_mode.py`

In [ ]:
base = evaluate(test_acc, max_new_tokens=512, etiket="[taban] ")
print("\n--- ornek cikti ---")
print(base["outs"][0][:600])

## 09 lora ve tokenizasyon

`hucreler/v1/09_lora_ve_tokenizasyon.py`

In [ ]:
from datasets import Dataset

if not hasattr(model, "peft_config"):
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=SEED,
    )
model.print_trainable_parameters()

def encode(row):
    p = tok(build_prompt(row["question"]), add_special_tokens=False)["input_ids"]
    c = tok(row["answer"] + END, add_special_tokens=False)["input_ids"]
    ids    = (p + c)[:MAX_SEQ_LEN]
    labels = ([-100] * len(p) + c)[:MAX_SEQ_LEN]
    return {"input_ids": ids, "labels": labels, "uzun": len(p) >= MAX_SEQ_LEN}

def to_ds(rows):
    ds = Dataset.from_list(rows).map(encode, remove_columns=["question","answer","letter","group"])
    atilan = sum(ds["uzun"])
    ds = ds.filter(lambda x: not x["uzun"]).remove_columns(["uzun"])
    return ds.map(lambda x: {"length": len(x["input_ids"])}), atilan

train_ds, a1 = to_ds(train_rows)
val_ds,   a2 = to_ds(val_rows)

lens = sorted(train_ds["length"])
maskesiz = sum(l != -100 for l in train_ds[0]["labels"]) / len(train_ds[0]["labels"])
print(f"train {len(train_ds)} | val {len(val_ds)} | promptu sigmayan atildi: {a1+a2}")
print(f"token p50 {lens[len(lens)//2]} | p95 {lens[int(len(lens)*.95)]} | max {lens[-1]}")
print(f"loss alan token orani (ilk ornek): {maskesiz:.0%}")

## 10 egitim

`hucreler/v1/10_egitim.py`

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = lambda s, v: s + [v] * (n - len(s))
    return {
        "input_ids":      torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
        "attention_mask": torch.tensor([pad([1]*len(b["input_ids"]), 0)       for b in batch]),
        "labels":         torch.tensor([pad(b["labels"], -100)                for b in batch]),
    }

adim = 20 if SMOKE else 200

import inspect
_kabul = set(inspect.signature(TrainingArguments.__init__).parameters)
_ek = {}
if "group_by_length" in _kabul:
    _ek["group_by_length"] = True
elif "train_sampling_strategy" in _kabul:
    _ek["train_sampling_strategy"] = "group_by_length"
if "warmup_ratio" in _kabul:
    _ek["warmup_ratio"] = 0.03
elif "warmup_steps" in _kabul:
    _ek["warmup_steps"] = 30
_eval_anahtar = "eval_strategy" if "eval_strategy" in _kabul else "evaluation_strategy"
_ek[_eval_anahtar] = "steps"
print("transformers uyumu:", {k: v for k, v in _ek.items()})

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=BS,
    gradient_accumulation_steps=GA,
    num_train_epochs=EPOCHS,
    max_steps=40 if SMOKE else -1,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.0,
    bf16=not LOAD_4BIT, fp16=LOAD_4BIT,
    optim="adamw_8bit",
    logging_steps=10,
    eval_steps=adim,
    save_strategy="steps", save_steps=adim, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    remove_unused_columns=False, report_to="none", seed=SEED,
    **_ek,
)

FastLanguageModel.for_training(model)
tok.padding_side = "right"
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collate)

son = get_last_checkpoint(OUT_DIR) if os.path.isdir(OUT_DIR) else None
print("devam edilen checkpoint:", son)
trainer.train(resume_from_checkpoint=son)

## 11 egitim sonrasi olcum

`hucreler/v1/11_egitim_sonrasi_olcum.py`

In [ ]:
ft = evaluate(test_acc, max_new_tokens=64, etiket="[sonra] ")

print(f"""
{'='*58}
                    TABAN      SONRA     FARK
dogruluk (gevsek)   {base['loose']:>6.1%}   {ft['loose']:>6.1%}   {ft['loose']-base['loose']:+.1%}
dogruluk (kati)     {base['strict']:>6.1%}   {ft['strict']:>6.1%}   {ft['strict']-base['strict']:+.1%}
bicime uyum         {base['format']:>6.1%}   {ft['format']:>6.1%}   {ft['format']-base['format']:+.1%}
{'='*58}
{len(test_acc)} soru. Gevsek satir asil bilgi kazancini, bicim satiri
bicim ogrenmesini gosterir. Ikisi birlikte okunmali.""")

## 12 anlamlilik testi

`hucreler/v1/12_anlamlilik_testi.py`

In [ ]:
from scipy.stats import binomtest

b_ok = [parse_loose(o) == r["letter"] for o, r in zip(base["outs"], test_acc)]
f_ok = [parse_loose(o) == r["letter"] for o, r in zip(ft["outs"],   test_acc)]
sadece_ft   = sum(f and not b for b, f in zip(b_ok, f_ok))
sadece_taban= sum(b and not f for b, f in zip(b_ok, f_ok))
n = sadece_ft + sadece_taban
p = binomtest(sadece_ft, n, 0.5).pvalue if n else 1.0
print(f"yalniz ince ayar dogru: {sadece_ft} | yalniz taban dogru: {sadece_taban} | ayrisma {n}")
print(f"McNemar (exact) p = {p:.3f}" + ("  -> anlamli" if p < 0.05 else "  -> anlamli degil"))

## 13 aciklama kalitesi

`hucreler/v1/13_aciklama_kalitesi.py`

In [ ]:
uzun = evaluate(test_expl[:5], max_new_tokens=400, etiket="[aciklama] ")
for r, o in zip(test_expl[:3], uzun["outs"]):
    print("="*78)
    print("SORU   :", r["question"].split("\n")[0][:110])
    print("GOLD   :", r["letter"])
    print("MODEL  :", o.strip()[:450])
    print("KITABIN:", r["reference"][:280])

## 14 kaydet

`hucreler/v1/14_kaydet.py`

In [ ]:
kayit = f"{OUT_DIR}/final_adapter"
model.save_pretrained(kayit)
tokenizer.save_pretrained(kayit)

sonuc = {
    "model": MODEL_NAME, "smoke": SMOKE, "epochs": EPOCHS, "seed": SEED,
    "train_size": len(train_ds), "val_size": len(val_ds),
    "test_size": len(test_acc),
    "baseline": {k: base[k] for k in ("strict","loose","format")},
    "finetuned": {k: ft[k]   for k in ("strict","loose","format")},
    "mcnemar": {"only_ft": sadece_ft, "only_base": sadece_taban, "p": p},
    "per_group_finetuned": {g: list(v) for g, v in ft["per_group"].items()},
}
with open(f"{OUT_DIR}/sonuc.json", "w", encoding="utf-8") as f:
    json.dump(sonuc, f, ensure_ascii=False, indent=2)

print("adapter:", kayit)
print(json.dumps({k: v for k, v in sonuc.items() if k != "per_group_finetuned"},
                 ensure_ascii=False, indent=2))

## 15 modeli indir

`hucreler/v1/15_modeli_indir.py`

In [ ]:
import json, shutil, os
from pathlib import Path
from google.colab import files

KAYNAK = "/content/drive/MyDrive/tus_sft/qwen35_4b/final_adapter"
SONUC  = "/content/drive/MyDrive/tus_sft/qwen35_4b/sonuc.json"
PAKET  = "/content/model"

assert os.path.isdir(KAYNAK), f"adapter bulunamadi: {KAYNAK}"
shutil.rmtree(PAKET, ignore_errors=True)
shutil.copytree(KAYNAK, PAKET)
if os.path.exists(SONUC):
    shutil.copy(SONUC, f"{PAKET}/sonuc.json")

KULLANIM = """# İnce ayarlı TUS modeli

Qwen3.5 4B üzerine eğitilmiş LoRA adapter. Temel model dahil değildir —
yüklerken `Qwen/Qwen3.5-4B` indirilir, adapter onun üzerine bindirilir.

## Yükleme

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3.5-4B", max_seq_length=2048, dtype="bfloat16",
)
model.load_adapter("model")
FastLanguageModel.for_inference(model)
```

## Soru sorma

```python
SYSTEM = ("Sen bir tıp uzmanısın. Aşağıdaki TUS sorusunu çöz. "
          "İlk satıra 'Doğru cevap: <harf>) <şık metni>' yaz, "
          "ardından kısa bir açıklama ver.")

soru = "Aşağıdakilerden hangisi...?\\nA) ...\\nB) ...\\nC) ...\\nD) ...\\nE) ..."
msgs = [{"role": "system", "content": SYSTEM},
        {"role": "user", "content": soru}]
p = tokenizer.apply_chat_template(msgs, tokenize=False,
                                  add_generation_prompt=True, enable_thinking=False)
ids = tokenizer(p, return_tensors="pt", add_special_tokens=False).to(model.device)
out = model.generate(**ids, max_new_tokens=400, do_sample=False, repetition_penalty=1.1)
print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))
```

`repetition_penalty=1.1` önemli — onsuz model bazı cümleleri tekrarlıyor.

## Sonuçlar

`sonuc.json` dosyasında: taban %62,3 → ince ayar %70,9 (+8,5 puan, p=0,003),
316 soruluk kontrol edilmiş test setinde.
"""
Path(f"{PAKET}/KULLANIM.md").write_text(KULLANIM, encoding="utf-8")

print("paket icerigi:")
for f in sorted(os.listdir(PAKET)):
    mb = os.path.getsize(f"{PAKET}/{f}") / 1e6
    print(f"  {f:<34} {mb:7.2f} MB")

shutil.make_archive("/content/model", "zip", PAKET)
print(f"\nzip: {os.path.getsize('/content/model.zip')/1e6:.1f} MB")
files.download("/content/model.zip")

## 16 hata dok

`hucreler/v1/16_hata_dok.py`

In [ ]:
import json, re
from google.colab import files

try:
    base_outs, ft_outs, test_acc
except NameError:
    raise SystemExit("Oturum dusmus — 01-11 arasi hucreleri yeniden calistir, sonra buraya don.")

def sik_metni(soru, harf):
    m = re.search(rf"(?m)^{harf}\) (.+)$", soru)
    return m.group(1).strip() if m else ""

kayitlar = []
for r, bo, fo in zip(test_acc, base_outs, ft_outs):
    bp, fp = parse_loose(bo), parse_loose(fo)
    g = r["letter"]
    kayitlar.append({
        "soru": r["question"],
        "kaynak": r.get("group", "?"),
        "dogru_harf": g,
        "dogru_metin": sik_metni(r["question"], g),
        "taban_harf": bp,
        "taban_metin": sik_metni(r["question"], bp) if bp else "",
        "taban_cikti": bo.strip()[:400],
        "ft_harf": fp,
        "ft_metin": sik_metni(r["question"], fp) if fp else "",
        "ft_cikti": fo.strip()[:400],
        "taban_dogru": bp == g,
        "ft_dogru": fp == g,
        "durum": ("ikisi_de_dogru" if bp == g and fp == g else
                  "ikisi_de_yanlis" if bp != g and fp != g else
                  "ft_kazandi" if fp == g else "ft_kaybetti"),
    })

from collections import Counter
print("durum dagilimi:", dict(Counter(k["durum"] for k in kayitlar)))
print("ft yanlis toplam:", sum(not k["ft_dogru"] for k in kayitlar))

with open("/content/hata_analizi.json", "w", encoding="utf-8") as f:
    json.dump(kayitlar, f, ensure_ascii=False, indent=1)
files.download("/content/hata_analizi.json")

## 17 hata analizi tek parca

`hucreler/v1/17_hata_analizi_tek_parca.py`

In [ ]:
import json, re, random, zipfile, glob, os, torch
from pathlib import Path
from collections import Counter
from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)

ZIP_PATH = "/content/drive/MyDrive/tus_data/TUS_Dataset_Final-20261002T154401Z-1-001.zip"
ADAPTER  = "/content/drive/MyDrive/tus_sft/qwen35_4b/final_adapter"
HEDEF    = "/content/tus_veri"
MODEL_NAME, MAX_SEQ_LEN, SEED = "Qwen/Qwen3.5-4B", 2048, 42

SYSTEM_PROMPT = ("Sen bir tıp uzmanısın. Aşağıdaki TUS sorusunu çöz. "
                 "İlk satıra 'Doğru cevap: <harf>) <şık metni>' yaz, ardından kısa bir açıklama ver.")

if not glob.glob(f"{HEDEF}/**/egitim.jsonl", recursive=True):
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(HEDEF)
DATA_DIR = str(Path(glob.glob(f"{HEDEF}/**/egitim.jsonl", recursive=True)[0]).parent)
assert os.path.isdir(ADAPTER), f"adapter yok: {ADAPTER}"

ANSWER = re.compile(r"Doğru cevap:\s*([A-E])")
test_acc = [{"question": r["soru"], "letter": r["dogru_cevap"],
             "group": (r.get("meta") or {}).get("group", "?")}
            for r in (json.loads(l) for l in open(f"{DATA_DIR}/test_cevap.jsonl", encoding="utf-8"))]
print(f"test: {len(test_acc)} soru | DATA_DIR: {DATA_DIR}")

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=torch.bfloat16, load_in_4bit=False)
tok = getattr(tokenizer, "tokenizer", tokenizer)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def build_prompt(q):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

BASLIK = re.compile(r"^\s*Doğru cevap\s*:?\s*\(?([A-E])\)?")
HERHANGI = re.compile(r"(?:Doğru\s*)?(?:cevap|yanıt|answer)\s*[:\-]?\s*\(?([A-E])\)?", re.I)
def parse_loose(t):
    c = re.sub(r"<think>.*?</think>", "", t, flags=re.S).strip()
    m = BASLIK.match(c) or HERHANGI.search(c) or re.search(r"\b([A-E])\)", c)
    return m.group(1).upper() if m else None

@torch.no_grad()
def uret(rows, max_new_tokens, bs=16, etiket=""):
    FastLanguageModel.for_inference(model)
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(rows), bs):
        b = rows[i:i+bs]
        enc = tok([build_prompt(r["question"]) for r in b], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"  {etiket}{min(i+bs,len(rows))}/{len(rows)}", end="\r")
    acc = sum(parse_loose(o) == r["letter"] for o, r in zip(outs, rows)) / len(rows)
    print(f"\n  {etiket}dogruluk {acc:.1%}")
    return outs, acc

print("\n[1/2] taban model olculuyor (uzun, ~12 dk)")
base_outs, base_acc = uret(test_acc, 512, etiket="[taban] ")

print("\n[2/2] adapter yukleniyor")
from peft import PeftModel
model = PeftModel.from_pretrained(model, ADAPTER)
ft_outs, ft_acc = uret(test_acc, 64, etiket="[ft] ")

def sik(soru, harf):
    m = re.search(rf"(?m)^{harf}\) (.+)$", soru)
    return m.group(1).strip() if m else ""

kayitlar = []
for r, bo, fo in zip(test_acc, base_outs, ft_outs):
    bp, fp, g = parse_loose(bo), parse_loose(fo), r["letter"]
    kayitlar.append({"soru": r["question"], "kaynak": r.get("group", "?"),
        "dogru_harf": g, "dogru_metin": sik(r["question"], g),
        "taban_harf": bp, "taban_metin": sik(r["question"], bp) if bp else "",
        "ft_harf": fp, "ft_metin": sik(r["question"], fp) if fp else "",
        "ft_cikti": fo.strip()[:400], "taban_dogru": bp == g, "ft_dogru": fp == g,
        "durum": ("ikisi_de_dogru" if bp == g and fp == g else
                  "ikisi_de_yanlis" if bp != g and fp != g else
                  "ft_kazandi" if fp == g else "ft_kaybetti")})

print("\ndurum:", dict(Counter(k["durum"] for k in kayitlar)))
print(f"taban {base_acc:.1%} | ft {ft_acc:.1%}")
with open("/content/hata_analizi.json", "w", encoding="utf-8") as f:
    json.dump(kayitlar, f, ensure_ascii=False, indent=1)
files.download("/content/hata_analizi.json")